# Solution: Implement Transformer with Group Query Attention (GQA) and Rotary Position Embeddings (RoPE)

This notebook provides a complete implementation comparing a Transformer with GQA+RoPE against a vanilla Transformer baseline on WikiText-2.

## Step 1: Data Acquisition

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import math
import time
import numpy as np
from torchtext.datasets import WikiText2
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from collections import Counter

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load WikiText-2 dataset
train_iter = WikiText2(split='train')
val_iter = WikiText2(split='valid')
test_iter = WikiText2(split='test')

# Initialize tokenizer
tokenizer = get_tokenizer('basic_english')

# Build vocabulary
def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)

vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=['<unk>', '<pad>'])
vocab.set_default_index(vocab['<unk>'])

print(f"Vocabulary size: {len(vocab)}")

## Step 2: Data Preprocessing

In [ ]:
def data_process(raw_text_iter, vocab, tokenizer):
    """Convert raw text to tensor of token indices"""
    data = [torch.tensor(vocab(tokenizer(item)), dtype=torch.long) for item in raw_text_iter]
    return torch.cat(tuple(filter(lambda t: t.numel() > 0, data)))

# Process datasets
train_iter, val_iter, test_iter = WikiText2()
train_data = data_process(train_iter, vocab, tokenizer)
val_data = data_process(val_iter, vocab, tokenizer)
test_data = data_process(test_iter, vocab, tokenizer)

print(f"Train data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Test data size: {len(test_data)}")

def batchify(data, batch_size):
    """Divide data into batches"""
    num_batches = data.size(0) // batch_size
    data = data[:num_batches * batch_size]
    data = data.view(batch_size, num_batches).t().contiguous()
    return data.to(device)

batch_size = 20
eval_batch_size = 10

train_data = batchify(train_data, batch_size)
val_data = batchify(val_data, eval_batch_size)
test_data = batchify(test_data, eval_batch_size)

def get_batch(source, i, seq_len=35):
    """Get a batch of data for training"""
    seq_len = min(seq_len, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].reshape(-1)
    return data, target

## Step 3: Implement Rotary Position Embeddings (RoPE)

In [ ]:
class RotaryPositionEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base
        
        # Precompute frequencies
        inv_freq = 1.0 / (self.base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        
        # Cache sin and cos for efficiency
        self._seq_len_cached = None
        self._cos_cached = None
        self._sin_cached = None
    
    def _update_cache(self, seq_len, device):
        """Update cached sin/cos values if sequence length changed"""
        if seq_len != self._seq_len_cached:
            self._seq_len_cached = seq_len
            t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
            freqs = torch.outer(t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1)
            self._cos_cached = emb.cos()[None, :, None, :]
            self._sin_cached = emb.sin()[None, :, None, :]
    
    def rotate_half(self, x):
        """Rotate half the hidden dims of the input"""
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat((-x2, x1), dim=-1)
    
    def apply_rotary_pos_emb(self, q, k):
        """Apply rotary embeddings to queries and keys"""
        seq_len = q.shape[1]
        self._update_cache(seq_len, q.device)
        
        q_embed = (q * self._cos_cached[:, :seq_len]) + (self.rotate_half(q) * self._sin_cached[:, :seq_len])
        k_embed = (k * self._cos_cached[:, :seq_len]) + (self.rotate_half(k) * self._sin_cached[:, :seq_len])
        
        return q_embed, k_embed

## Step 4: Implement Group Query Attention (GQA)

In [ ]:
class GroupQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads  # Number of query heads per kv head
        
        # Query projection (full heads)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        
        # Key and Value projections (grouped heads)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        # Initialize RoPE
        self.rope = RotaryPositionEmbedding(self.head_dim)
        
        self.scale = self.head_dim ** -0.5
    
    def repeat_kv(self, x):
        """Repeat key/value heads to match number of query heads"""
        batch_size, seq_len, n_kv_heads, head_dim = x.shape
        if self.n_rep == 1:
            return x
        return x[:, :, :, None, :].expand(batch_size, seq_len, n_kv_heads, self.n_rep, head_dim).reshape(
            batch_size, seq_len, n_kv_heads * self.n_rep, head_dim
        )
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim)
        
        # Apply RoPE
        q, k = self.rope.apply_rotary_pos_emb(q, k)
        
        # Repeat k and v to match number of query heads
        k = self.repeat_kv(k)
        v = self.repeat_kv(v)
        
        # Transpose for attention computation: (batch, n_heads, seq_len, head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # Compute attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        
        # Apply mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Apply softmax and dropout
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention to values
        out = torch.matmul(attn, v)
        
        # Reshape and project output
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        out = self.out_proj(out)
        
        return out

## Step 4b: Implement Vanilla Multi-Head Attention for Baseline

In [ ]:
class VanillaMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        out = self.out_proj(out)
        
        return out

## Step 5: Build Complete Transformer Models

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, dropout=0.1, use_gqa=True):
        super().__init__()
        
        if use_gqa:
            self.attention = GroupQueryAttention(d_model, n_heads, n_kv_heads, dropout)
        else:
            self.attention = VanillaMultiHeadAttention(d_model, n_heads, dropout)
        
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_out = self.attention(self.norm1(x), mask)
        x = x + self.dropout(attn_out)
        
        # Feed-forward with residual connection
        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout(ff_out)
        
        return x

class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, 
                 max_seq_len, dropout=0.1, use_gqa=True):
        super().__init__()
        
        self.d_model = d_model
        self.use_gqa = use_gqa
        
        # Token embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional embedding (only for vanilla, GQA uses RoPE)
        if not use_gqa:
            self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, n_kv_heads, d_ff, dropout, use_gqa)
            for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.output_projection = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, x, mask=None):
        batch_size, seq_len = x.shape
        
        # Token embeddings
        x = self.token_embedding(x) * math.sqrt(self.d_model)
        
        # Add positional embeddings for vanilla transformer
        if not self.use_gqa:
            positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
            x = x + self.pos_embedding(positions)
        
        x = self.dropout(x)
        
        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x, mask)
        
        x = self.norm(x)
        logits = self.output_projection(x)
        
        return logits

def generate_square_subsequent_mask(sz, device):
    """Generate causal mask for autoregressive modeling"""
    mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask

## Step 6: Training Setup

In [ ]:
def count_parameters(model):
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_epoch(model, train_data, optimizer, criterion, seq_len, device):
    model.train()
    total_loss = 0.
    start_time = time.time()
    
    num_batches = len(train_data) - 1
    
    for batch_idx, i in enumerate(range(0, train_data.size(0) - 1, seq_len)):
        data, targets = get_batch(train_data, i, seq_len)
        
        # Create causal mask
        mask = generate_square_subsequent_mask(data.size(0), device).unsqueeze(0).unsqueeze(0)
        
        optimizer.zero_grad()
        
        output = model(data.t(), mask)  # Transpose to (batch, seq_len)
        output = output.view(-1, len(vocab))
        
        loss = criterion(output, targets)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 200 == 0 and batch_idx > 0:
            cur_loss = total_loss / 200
            elapsed = time.time() - start_time
            print(f'| batch {batch_idx:5d}/{num_batches:5d} | '
                  f'lr {optimizer.param_groups[0]["lr"]:02.6f} | '
                  f'ms/batch {elapsed * 1000 / 200:5.2f} | '
                  f'loss {cur_loss:5.2f} | ppl {math.exp(cur_loss):8.2f}')
            total_loss = 0
            start_time = time.time()
    
    return total_loss / num_batches

def evaluate(model, eval_data, criterion, seq_len, device):
    model.eval()
    total_loss = 0.
    
    with torch.no_grad():
        for i in range(0, eval_data.size(0) - 1, seq_len):
            data, targets = get_batch(eval_data, i, seq_len)
            mask = generate_square_subsequent_mask(data.size(0), device).unsqueeze(0).unsqueeze(0)
            
            output = model(data.t(), mask)
            output = output.view(-1, len(vocab))
            total_loss += criterion(output, targets).item() * data.size(0)
    
    return total_loss / (len(eval_data) - 1)

## Step 6b: Initialize and Train Models

In [ ]:
# Model hyperparameters
vocab_size = len(vocab)
d_model = 256
n_layers = 4
n_heads = 8
n_kv_heads = 4  # For GQA (2 query heads share 1 kv head)
d_ff = 1024
max_seq_len = 512
dropout = 0.2
seq_len = 35

# Initialize models
print("\n=== Initializing GQA + RoPE Model ===")
gqa_model = TransformerLM(
    vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, 
    max_seq_len, dropout, use_gqa=True
).to(device)

print("\n=== Initializing Vanilla Transformer Model ===")
vanilla_model = TransformerLM(
    vocab_size, d_model, n_layers, n_heads, n_heads, d_ff,  # n_kv_heads = n_heads for vanilla
    max_seq_len, dropout, use_gqa=False
).to(device)

# Print parameter counts
gqa_params = count_parameters(gqa_model)
vanilla_params = count_parameters(vanilla_model)
print(f"\nGQA Model Parameters: {gqa_params:,}")
print(f"Vanilla Model Parameters: {vanilla_params:,}")
print(f"Parameter Reduction: {(1 - gqa_params/vanilla_params)*100:.2f}%")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
lr = 0.0005

gqa_optimizer = torch.optim.AdamW(gqa_model.parameters(), lr=lr)
vanilla_optimizer = torch.optim.AdamW(vanilla_model.parameters(), lr=lr)

# Learning rate scheduler
gqa_scheduler = torch.optim.lr_scheduler.StepLR(gqa_optimizer, 1.0, gamma=0.95)
vanilla_scheduler = torch.optim.lr_scheduler.StepLR(vanilla_optimizer, 1.0, gamma=0.95)

In [ ]:
# Training loop
epochs = 10

gqa_train_losses = []
gqa_val_losses = []
gqa_times = []

vanilla_train_losses = []
vanilla_val_losses = []
vanilla_times = []

print("\n" + "="*50)
print("Training GQA + RoPE Model")
print("="*50)

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()
    print(f'\n| Epoch {epoch} |')
    
    train_epoch(gqa_model, train_data, gqa_optimizer, criterion, seq_len, device)
    val_loss = evaluate(gqa_model, val_data, criterion, seq_len, device)
    
    elapsed = time.time() - epoch_start_time
    gqa_times.append(elapsed)
    gqa_val_losses.append(val_loss)
    
    print('-' * 89)
    print(f'| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | '
          f'valid loss {val_loss:5.2f} | valid ppl {math.exp(val_loss):8.2f}')
    print('-' * 89)
    
    gqa_scheduler.step()

print("\n" + "="*50)
print("Training Vanilla Transformer Model")
print("="*50)

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()
    print(f'\n| Epoch {epoch} |')
    
    train_epoch(vanilla_model, train_data, vanilla_optimizer, criterion, seq_len, device)
    val_loss = evaluate(vanilla_model, val_data, criterion, seq_len, device)
    
    elapsed = time.time() - epoch_start_time
    vanilla_times.append(elapsed)
    vanilla_val_losses.append(val_loss)
    
    print('-' * 89)
    print(f'| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | '
          f'valid loss {val_loss:5.2f} | valid ppl {math.exp(val_loss):8.2f}')
    print('-' * 89)
    
    vanilla_scheduler.step()

## Step 7: Model Comparison

In [ ]:
# Final evaluation on test set
gqa_test_loss = evaluate(gqa_model, test_data, criterion, seq_len, device)
vanilla_test_loss = evaluate(vanilla_model, test_data, criterion, seq_len, device)

gqa_test_ppl = math.exp(gqa_test_loss)
vanilla_test_ppl = math.exp(vanilla_test_loss)

print("\n" + "="*50)
print("FINAL COMPARISON")
print("="*50)

comparison_data = {
    'Metric': ['Parameters', 'Avg Training Time (s/epoch)', 'Test Loss', 'Test Perplexity'],
    'GQA + RoPE': [
        f"{gqa_params:,}",
        f"{np.mean(gqa_times):.2f}",
        f"{gqa_test_loss:.4f}",
        f"{gqa_test_ppl:.2f}"
    ],
    'Vanilla': [
        f"{vanilla_params:,}",
        f"{np.mean(vanilla_times):.2f}",
        f"{vanilla_test_loss:.4f}",
        f"{vanilla_test_ppl:.2f}"
    ],
    'Improvement': [
        f"{(1 - gqa_params/vanilla_params)*100:.2f}% fewer",
        f"{(1 - np.mean(gqa_times)/np.mean(vanilla_times))*100:.2f}% faster" if np.mean(gqa_times) < np.mean(vanilla_times) else f"{(np.mean(gqa_times)/np.mean(vanilla_times) - 1)*100:.2f}% slower",
        f"{(vanilla_test_loss - gqa_test_loss):.4f}" if gqa_test_loss < vanilla_test_loss else f"+{(gqa_test_loss - vanilla_test_loss):.4f}",
        f"{(vanilla_test_ppl - gqa_test_ppl):.2f}" if gqa_test_ppl < vanilla_test_ppl else f"+{(gqa_test_ppl - vanilla_test_ppl):.2f}"
    ]
}

print("\nComparison Table:")
print("-" * 100)
print(f"{'Metric':<30} {'GQA + RoPE':<20} {'Vanilla':<20} {'Improvement':<20}")
print("-" * 100)
for i in range(len(comparison_data['Metric'])):
    print(f"{comparison_data['Metric'][i]:<30} {comparison_data['GQA + RoPE'][i]:<20} "
          f"{comparison_data['Vanilla'][i]:<20} {comparison_data['Improvement'][i]:<20}")
print("-" * 100)

# Speed comparison for inference
print("\n" + "="*50)
print("Inference Speed Test")
print("="*50)

test_batch, _ = get_batch(test_data, 0, seq_len)
mask = generate_square_subsequent_mask(test_batch.size(0), device).unsqueeze(0).unsqueeze(0)

# GQA inference time
gqa_model.eval()
gqa_inference_times = []
with torch.no_grad():
    for _ in range(100):
        start = time.time()
        _ = gqa_model(test_batch.t(), mask)
        gqa_inference_times.append(time.time() - start)

# Vanilla inference time
vanilla_model.eval()
vanilla_inference_times = []
with torch.no_grad():
    for _ in range(100):
        start = time.time()
        _ = vanilla_model(test_batch.t(), mask)
        vanilla_inference_times.append(time.time() - start)

gqa_avg_inference = np.mean(gqa_inference_times) * 1000  # Convert to ms
vanilla_avg_inference = np.mean(vanilla_inference_times) * 1000

print(f"GQA + RoPE avg inference time: {gqa_avg_inference:.2f} ms")
print(f"Vanilla avg inference time: {vanilla_avg_inference:.2f} ms")
print(f"Speedup: {vanilla_avg_inference / gqa_avg_inference:.2f}x")

## Step 8: Analysis and Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Validation Perplexity over epochs
gqa_val_ppl = [math.exp(loss) for loss in gqa_val_losses]
vanilla_val_ppl = [math.exp(loss) for loss in vanilla_val_losses]

axes[0].plot(range(1, epochs + 1), gqa_val_ppl, 'b-o', label='GQA + RoPE', linewidth=2)
axes[0].plot(range(1, epochs + 1), vanilla_val_ppl, 'r-s', label='Vanilla', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Validation Perplexity', fontsize=12)
axes[0].set_title('Validation Perplexity Over Time', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Training time comparison
axes[1].bar(['GQA + RoPE', 'Vanilla'], 
           [np.mean(gqa_times), np.mean(vanilla_times)],
           color=['blue', 'red'], alpha=0.7)
axes[1].set_ylabel('Time (seconds)', fontsize=12)
axes[1].set_title('Average Training Time per Epoch', fontsize=14)
axes[1].grid(True, alpha=0.3, axis='y')

# Parameter comparison
axes[2].bar(['GQA + RoPE', 'Vanilla'], 
           [gqa_params/1e6, vanilla_params/1e6],
           color=['blue', 'red'], alpha=0.7)
axes[2].set_ylabel('Parameters (Millions)', fontsize=12)
axes[2].set_title('Model Size Comparison', fontsize=14)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary analysis
print("\n" + "="*50)
print("KEY INSIGHTS")
print("="*50)
print(f"""
1. Parameter Efficiency:
   - GQA reduces parameters by {(1 - gqa_params/vanilla_params)*100:.2f}% by using grouped key-value heads
   - This reduction comes from using {n_kv_heads} KV heads instead of {n_heads}

2. Speed:
   - Training: GQA is {'faster' if np.mean(gqa_times) < np.mean(vanilla_times) else 'slower'}
   - Inference: {vanilla_avg_inference / gqa_avg_inference:.2f}x speedup with GQA

3. Performance:
   - Test Perplexity: GQA={gqa_test_ppl:.2f}, Vanilla={vanilla_test_ppl:.2f}
   - {'GQA performs better' if gqa_test_ppl < vanilla_test_ppl else 'Vanilla performs better'} on this task

4. When to Use GQA + RoPE:
   - Memory-constrained environments (mobile, edge devices)
   - When inference speed is critical
   - Long-context applications (RoPE handles relative positions well)
   - When slight performance trade-offs are acceptable for efficiency gains

5. RoPE Benefits:
   - No learned positional embeddings (saves parameters)
   - Better extrapolation to longer sequences
   - Encodes relative position information naturally
""")